# Phase B (Notebook 2): Spoiler-Aware GraphRAG & 3-Round Agentic Retrieval Test (`PhaseB2_GraphRAG_Retrieval_Test.ipynb`)

**Environment Stack**: Runs on a **free Google Colab CPU runtime** (0 GPU compute units required). Connects `DeepSeek-V3` (`deepseek-chat`) + `deepseek-flash` (`inspect_keyframes` Vision Sub-Tool) to the pre-built 3-tier GraphRAG index in `/content/drive/MyDrive/MovieRAG/workdir/{MOVIE_ID}`.

### Workflow Overview
1. **Cell 1 — Google Drive Mount, `DEEPSEEK_API_KEY` & Pre-Built GraphRAG Index Check**:
   - Mounts Drive, loads `DEEPSEEK_API_KEY` from **Colab Secrets**, and verifies all 6 GraphRAG index artifacts in `WORKDIR`.
2. **Cell 2 — Clone / Pull Latest Code from GitHub (`/content/MovieRag_repo`)**:
   - Clones/pulls `main` and registers `MovieRAG/videorag` on `sys.path`.
3. **Cell 3 — Install Lightweight CPU Dependencies & Load `VideoRAG`**:
   - Installs lightweight dependencies and initializes `VideoRAG(working_dir=WORKDIR, llm=deepseek_config)` in `< 3 seconds`.
4. **Cell 4 — Inspect Raw Turn-0 Spoiler-Filtered Context (`only_need_context=True`)**:
   - Lets you inspect the exact 4-block XML Turn-0 context (`<viewer_state>`, `<current_scene_at_T_user>`, `<graph_entities_and_relations_up_to_T_user>`, `<retrieved_scene_and_chapter_chunks_up_to_T_user>`) at any `TIMESTAMP_MMSS` and verify straddle-safe subtitle/micro-beat clipping.
5. **Cell 5 — Spoiler-Isolation A/B Benchmark (`spoiler_free=True` vs `spoiler_free=False`)**:
   - Tests the same dragon-twist question across early (`02:00`), mid (`04:00`), and full-movie (`spoiler_free=False`) modes to verify strict spoiler protection.
6. **Cell 6 — Test the 4 Spoiler-Gated Agent Tools (`search_scenes`, `get_character_history`, `get_adjacent_scenes`, `inspect_keyframes`)**:
   - Runs queries that trigger tool calls (including on-demand Vision API keyframe inspection) and prints the step-by-step `last_tool_calls_trace`.
7. **Cell 7 — Interactive Multi-Turn Chat Playground (with Timeline Scrubber & `chat_history` Memory)**:
   - Ask follow-up questions with pronouns (*"Why did he say that?"*, *"Where did she find him?"*) while scrubbing `current_timestamp` (`MM:SS`) or toggling `spoiler_free`.

In [ ]:
# ==============================================================================
# CELL 1: Google Drive Mount, DeepSeek API Key & Pre-Built GraphRAG Index Check
# ==============================================================================
import os

try:
    from google.colab import drive, userdata
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not running inside Google Colab; using local paths.")

MOVIE_ID = "sintel"                              # @param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/MovieRAG"   # @param {type:"string"}
API_BASE_URL = "https://api.deepseek.com"        # @param {type:"string"}
VISION_MODEL_ID = "deepseek-flash"               # @param {type:"string"}
EMBEDDING_PROVIDER = "sentence_transformers"     # @param ["sentence_transformers", "auto", "siliconflow", "ollama", "hash"]
ST_EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"     # @param {type:"string"}
DEEPSEEK_API_KEY_OVERRIDE = ""                   # @param {type:"string"}

WORKDIR = f"{DRIVE_ROOT}/workdir/{MOVIE_ID}"

os.environ["DEEPSEEK_BASE_URL"] = API_BASE_URL
os.environ["VISION_MODEL_ID"] = VISION_MODEL_ID
os.environ["MOVIERAG_EMBEDDING_PROVIDER"] = EMBEDDING_PROVIDER
os.environ["MOVIERAG_ST_MODEL"] = ST_EMBEDDING_MODEL
os.environ["MOVIERAG_EMBEDDING_DIM"] = "768"

_api_key = DEEPSEEK_API_KEY_OVERRIDE.strip() or os.environ.get("DEEPSEEK_API_KEY", "").strip()
if not _api_key and IN_COLAB:
    try:
        _api_key = (userdata.get("DEEPSEEK_API_KEY") or "").strip()
    except Exception:
        _api_key = ""

if _api_key:
    os.environ["DEEPSEEK_API_KEY"] = _api_key
    print(f"[OK] DEEPSEEK_API_KEY loaded (starts with '{_api_key[:6]}...')")
else:
    print("[WARNING] DEEPSEEK_API_KEY not found! Add 'DEEPSEEK_API_KEY' in Colab Secrets or paste it into DEEPSEEK_API_KEY_OVERRIDE.")

for fname in [
    "movie_knowledge_bundle.json",
    "kv_store_video_segments.json",
    "kv_store_text_chunks.json",
    "vdb_chunks.json",
    "vdb_entities.json",
    "graph_chunk_entity_relation.graphml",
]:
    fpath = os.path.join(WORKDIR, fname)
    ok = os.path.isfile(fpath)
    print(f"  [{'OK' if ok else 'MISSING (Run PhaseB1 first)'}] {fname}: {fpath}")

In [ ]:
# ==============================================================================
# CELL 2: Clone / Pull Latest Source Code from GitHub (/content/MovieRag_repo)
# ==============================================================================
import os
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/ctu-flix/MovieRag.git"  # @param {type:"string"}
GITHUB_BRANCH = "main"                                        # @param {type:"string"}
CLONE_DIR = "/content/MovieRag_repo"

github_token = None
if globals().get("IN_COLAB", False):
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None

auth_repo_url = GITHUB_REPO_URL
if github_token and GITHUB_REPO_URL.startswith("https://github.com/"):
    auth_repo_url = GITHUB_REPO_URL.replace("https://github.com/", f"https://{github_token}@github.com/")

if not os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"Cloning {GITHUB_REPO_URL} (branch: {GITHUB_BRANCH})...")
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, auth_repo_url, CLONE_DIR], check=True)
else:
    print(f"Pulling latest commits from origin/{GITHUB_BRANCH}...")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", auth_repo_url], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "checkout", GITHUB_BRANCH], check=True)
    subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", GITHUB_BRANCH], check=True)

PROJECT_ROOT = CLONE_DIR if os.path.isdir(os.path.join(CLONE_DIR, "MovieRAG")) else os.path.join(CLONE_DIR, "MovieRag")
MOVIERAG_PKG_DIR = os.path.join(PROJECT_ROOT, "MovieRAG")

for p in (MOVIERAG_PKG_DIR, PROJECT_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

for mod_name in [k for k in list(sys.modules) if k == "videorag" or k.startswith("videorag.")]:
    del sys.modules[mod_name]

commit_log = subprocess.run(["git", "-C", CLONE_DIR, "log", "-1", "--oneline"], capture_output=True, text=True).stdout.strip()
print(f"Active MOVIERAG_PKG : {MOVIERAG_PKG_DIR}")
print(f"Latest Commit       : {commit_log}")

In [ ]:
# ==============================================================================
# CELL 3: Install Lightweight CPU Dependencies & Load Pre-Indexed VideoRAG
# ==============================================================================
import subprocess
import sys

pkgs = [
    "openai>=1.50.0",
    "httpx",
    "tenacity",
    "tiktoken",
    "networkx",
    "nano-vectordb",
    "sentence-transformers",
    "pillow",
    "nest_asyncio",
]
print("Installing dependencies:", ", ".join(pkgs))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed"] + pkgs,
    check=True,
)

import nest_asyncio
nest_asyncio.apply()

from videorag import VideoRAG, QueryParam, deepseek_config
from videorag._op import _parse_mmss_to_sec, _format_mmss

rag = VideoRAG(
    working_dir=WORKDIR,
    llm=deepseek_config,
)

graph = rag.chunk_entity_relation_graph._graph
print(f"\n[READY] Loaded VideoRAG from {WORKDIR}")
print(f"  - Movie Title    : {rag.addon_params.get('movie_title')}")
print(f"  - Duration       : {rag.addon_params.get('duration_sec')}s")
print(f"  - Text Chunks    : {len(rag.text_chunks._data)}")
print(f"  - Graph Nodes    : {graph.number_of_nodes()} | Edges: {graph.number_of_edges()}")

In [ ]:
# ==============================================================================
# CELL 4: Inspect Raw Turn-0 Spoiler-Filtered Context (only_need_context=True)
# ==============================================================================
# Use this cell to inspect the exact XML context block sent to DeepSeek-V3 at Turn 0
# and verify straddle-safe subtitle clipping, micro-beat clipping, and entity hiding.
import asyncio

TEST_QUERY = "What is happening right now and what is Sintel searching for?"  # @param {type:"string"}
TIMESTAMP_MMSS = "01:50"                                                      # @param {type:"string"}
SPOILER_FREE = True                                                           # @param {type:"boolean"}
TOP_K_CHUNKS = 5                                                              # @param {type:"integer"}

t_sec = _parse_mmss_to_sec(TIMESTAMP_MMSS)
turn0_xml = asyncio.get_event_loop().run_until_complete(
    rag.aquery(
        TEST_QUERY,
        param=QueryParam(
            mode="videorag",
            only_need_context=True,
            current_timestamp=t_sec,
            spoiler_free=SPOILER_FREE,
            top_k=TOP_K_CHUNKS,
        ),
    )
)

print(f"=== TURN-0 PRE-RETRIEVED XML CONTEXT (T_user={TIMESTAMP_MMSS} / {t_sec}s | spoiler_free={SPOILER_FREE}) ===\n")
print(turn0_xml)

In [ ]:
# ==============================================================================
# CELL 5: Spoiler-Isolation A/B Benchmark (spoiler_free=True vs spoiler_free=False)
# ==============================================================================
import asyncio
import time

BENCHMARK_QUERY = "What happens to Scales the baby dragon, and what is the truth about the Adult Dragon?"  # @param {type:"string"}

scenarios = [
    ("1. Early Watch (02:00 / 120s) — Spoiler-Free ON", 120.0, True),
    ("2. Post-Flashback (05:40 / 340s) — Spoiler-Free ON", 340.0, True),
    ("3. Full-Movie Mode (02:00 / 120s) — Spoiler-Free OFF", 120.0, False),
]

for title, t_user, sf in scenarios:
    print("=" * 88)
    print(f"{title} (T_user={_format_mmss(t_user)}, spoiler_free={sf})")
    print("=" * 88)
    t0 = time.perf_counter()
    ans = asyncio.get_event_loop().run_until_complete(
        rag.aquery(
            BENCHMARK_QUERY,
            param=QueryParam(
                mode="videorag",
                current_timestamp=t_user,
                spoiler_free=sf,
                max_tool_rounds=3,
            ),
        )
    )
    elapsed = time.perf_counter() - t0
    trace = rag.addon_params.get("last_tool_calls_trace", [])
    print(f"[Latency: {elapsed:.1f}s | Tool Calls: {len(trace)}]")
    for tc in trace:
        print(f"  -> Round {tc['round']} Tool: {tc['tool']}({tc['arguments']})")
    print("\n" + ans + "\n")

In [ ]:
# ==============================================================================
# CELL 6: Test Agentic Tool-Calling & Vision Sub-Tool (inspect_keyframes)
# ==============================================================================
import asyncio
import time

# Prompt designed to encourage DeepSeek-V3 to inspect keyframes or walk adjacent scenes/character history
TOOL_TEST_QUERY = "Inspect the keyframes of the scene in the Shaman's hut around 01:45: what exact expression, face paint, or objects are visible on the Shaman and Sintel?"  # @param {type:"string"}
TOOL_TEST_TIMESTAMP = "02:30"  # @param {type:"string"}
TOOL_TEST_SPOILER_FREE = True  # @param {type:"boolean"}

t_sec = _parse_mmss_to_sec(TOOL_TEST_TIMESTAMP)
t0 = time.perf_counter()
answer = asyncio.get_event_loop().run_until_complete(
    rag.aquery(
        TOOL_TEST_QUERY,
        param=QueryParam(
            mode="videorag",
            current_timestamp=t_sec,
            spoiler_free=TOOL_TEST_SPOILER_FREE,
            max_tool_rounds=3,
        ),
    )
)
elapsed = time.perf_counter() - t0
trace = rag.addon_params.get("last_tool_calls_trace", [])

print(f"[Completed in {elapsed:.1f}s | Tool Calls Executed: {len(trace)}]")
for i, tc in enumerate(trace, start=1):
    print(f"\n--- Tool Call #{i} (Round {tc['round']}) ---")
    print(f"Tool      : {tc['tool']}")
    print(f"Arguments : {tc['arguments']}")
    print(f"Output    : {tc['output_preview']}...")

print("\n=== FINAL GROUNDED ANSWER ===\n")
print(answer)

In [ ]:
# ==============================================================================
# CELL 7: Multi-Turn Interactive Chat Playground (with Timeline Scrubber & chat_history)
# ==============================================================================
import asyncio
import time

CHAT_HISTORY = []

def reset_chat():
    global CHAT_HISTORY
    CHAT_HISTORY = []
    print("[Chat history cleared]")

def ask_movierag(question: str, timestamp_mmss: str = "02:30", spoiler_free: bool = True):
    global CHAT_HISTORY
    t_sec = _parse_mmss_to_sec(timestamp_mmss)
    print("-" * 88)
    print(f"USER [T_user={timestamp_mmss} ({t_sec}s) | spoiler_free={spoiler_free} | history_msgs={len(CHAT_HISTORY[-6:])}]:")
    print(f"  {question}")
    t0 = time.perf_counter()
    ans = asyncio.get_event_loop().run_until_complete(
        rag.aquery(
            question,
            param=QueryParam(
                mode="videorag",
                current_timestamp=t_sec,
                spoiler_free=spoiler_free,
                max_tool_rounds=3,
                chat_history=list(CHAT_HISTORY[-6:]),
            ),
        )
    )
    elapsed = time.perf_counter() - t0
    trace = rag.addon_params.get("last_tool_calls_trace", [])
    CHAT_HISTORY.append({"role": "user", "content": question})
    CHAT_HISTORY.append({"role": "assistant", "content": ans})
    CHAT_HISTORY = CHAT_HISTORY[-6:]

    print(f"\nMOVIERAG ({elapsed:.1f}s | {len(trace)} tool calls):")
    for tc in trace:
        print(f"  [Tool Round {tc['round']}] {tc['tool']}({tc['arguments']})")
    print(ans)
    return ans

# Demo 3-turn conversation where the user scrubs the video timeline forward between turns:
reset_chat()
ask_movierag("Who is the old man tending to Sintel in the hut?", timestamp_mmss="02:00", spoiler_free=True)
ask_movierag("Why did he say her weapon has a dark past, and what did she tell him she is looking for?", timestamp_mmss="02:20", spoiler_free=True)
ask_movierag("How did she first meet that little dragon?", timestamp_mmss="03:45", spoiler_free=True)